# Finite Difference Operators for Monge-Ampère

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from monge_ampere.operators import laplacian, generate_stencil_directions, \
  ma_operator, det_hessian_standard, directional_second_derivative
from monge_ampere.boundary import BoundaryCondition

## Standard 5-point Laplacian
Let's test the 5-point Laplacian on a smooth sinusoidal function $u(x, y) = \sin(2\pi x) \sin(2\pi y)$.
The analytical Laplacian is $-8\pi^2 \sin(2\pi x) \sin(2\pi y)$.

In [ ]:
n = 64
h = 1.0 / n
x = np.arange(n) * h
X, Y = np.meshgrid(x, x, indexing="ij")

u = np.sin(2 * np.pi * X) * np.sin(2 * np.pi * Y)
analytical_laplacian = -8 * np.pi**2 * u

computed_laplacian = laplacian(u, h, bc=BoundaryCondition.PERIODIC)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(analytical_laplacian.T, origin="lower", extent=[0, 1, 0, 1], cmap="viridis")
axes[0].set_title("Analytical Laplacian")
axes[1].imshow(computed_laplacian.T, origin="lower", extent=[0, 1, 0, 1], cmap="viridis")
axes[1].set_title("Computed Laplacian")
err = np.abs(analytical_laplacian - computed_laplacian)
im = axes[2].imshow(err.T, origin="lower", extent=[0, 1, 0, 1], cmap="magma")
axes[2].set_title("Absolute Error")
plt.colorbar(im, ax=axes[2])
plt.show()

## Stencil Directions for Monotone Discretization
The Oberman-Froese wide-stencil scheme generates orthogonal pairs of vectors within a certain radius `dw` to ensure monotonicity for the Monge-Ampère equation. Let's visualize the pairs for `dw = 2`.

In [ ]:
dw = 2
pairs = generate_stencil_directions(dw)

fig, ax = plt.subplots(figsize=(6, 6))
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)

for v, w in pairs:
    ax.arrow(0, 0, v[0], v[1], head_width=0.1, head_length=0.1, fc='blue', ec='blue', alpha=0.5)
    ax.arrow(0, 0, w[0], w[1], head_width=0.1, head_length=0.1, fc='red', ec='red', alpha=0.5)

ax.set_xlim(-dw-1, dw+1)
ax.set_ylim(-dw-1, dw+1)
ax.grid(True, linestyle='--', alpha=0.6)
ax.set_aspect('equal')
ax.set_title(f"Orthogonal Stencil Pairs (dw={dw})")
plt.show()

## Monotone Discretization of $\det(D^2u)$
The Monge-Ampère expression is evaluated via the minimum over all generated orthogonal direction pairs:
$$ \det(D^2u) \approx \min_{v \perp w} (D_{vv}u)^+ (D_{ww}u)^+ $$
Let's verify it on $u(x,y) = \frac{1}{2}(x^2 + y^2)$, where the analytical determinant is 1.

In [ ]:
# Note: For testing the standard monotone operator, we can just use 
# a local quadratic bump that doesn't need periodic wrapping, 
# or solve using standard Dirichlet bounds.
X, Y = np.meshgrid(np.linspace(-1, 1, 32), np.linspace(-1, 1, 32), indexing="ij")
h_bump = 2.0 / 31
u_bump = 0.5 * (X**2 + Y**2)

det_monotone = ma_operator(u_bump, h_bump, dw=2, bc=BoundaryCondition.DIRICHLET, boundary_vals=u_bump)
det_standard = det_hessian_standard(u_bump, h_bump, bc=BoundaryCondition.DIRICHLET, boundary_vals=u_bump)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im0 = axes[0].imshow(det_monotone.T, origin="lower", extent=[-1, 1, -1, 1], vmin=0.9, vmax=1.1)
axes[0].set_title(r"Monotone MA Evaluated on $\frac{1}{2}(x^2+y^2)$")
plt.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(det_standard.T, origin="lower", extent=[-1, 1, -1, 1], vmin=0.9, vmax=1.1)
axes[1].set_title(r"Standard Det Hessian Evaluated on $\frac{1}{2}(x^2+y^2)$")
plt.colorbar(im1, ax=axes[1])
plt.show()